# FINN Shape Debug Notebook

Dieses Notebook lädt ein ONNX-Modell (`temp_model.onnx`), führt Shape-Inference durch,
listet alle Shape-Informationen auf und sucht gezielt nach Konflikten wie `80` vs `45`.

⚠️ Stelle sicher, dass `temp_model.onnx` im gleichen Verzeichnis liegt.

In [1]:
import onnx
from onnx import shape_inference
from onnx import numpy_helper
import numpy as np
from pprint import pprint

MODEL_PATH = "temp_model.onnx"

## 1. Modell laden

In [2]:
model = onnx.load(MODEL_PATH)
print("Modell geladen.")
print("Nodes im Modell:", len(model.graph.node))

Modell geladen.
Nodes im Modell: 58


## 2. ONNX-Shape-Inference ausführen
Wenn hier ein Fehler kommt, ist das Modell bereits vor FINN inkonsistent.

In [3]:
try:
    model_si = shape_inference.infer_shapes(model)
    onnx.save(model_si, "temp_model_inferred.onnx")
    print("Shape-Inference erfolgreich. Ergebnis gespeichert als temp_model_inferred.onnx")
except Exception as e:
    print("❌ Fehler bei der Shape-Inference:")
    print(e)
    raise

Shape-Inference erfolgreich. Ergebnis gespeichert als temp_model_inferred.onnx


## 3. Alle verfügbaren Shape-Informationen auflisten

In [4]:
def extract_shape(value_info):
    shape = []
    t = value_info.type.tensor_type
    for d in t.shape.dim:
        if d.dim_value > 0:
            shape.append(d.dim_value)
        else:
            shape.append(None)
    return shape

print("🔍 Shapes aus graph.value_info:")
for vi in model_si.graph.value_info:
    print(f"{vi.name}: {extract_shape(vi)}")

print("\n🔍 Input/Output-Shapes:")
for vi in list(model_si.graph.input) + list(model_si.graph.output):
    print(f"{vi.name}: {extract_shape(vi)}")

🔍 Shapes aus graph.value_info:
/quant_input/act_quant/export_handler/Quant_output_0: [1, 4, 360, 640]
/model/quant_input/act_quant/export_handler/Quant_output_0: [1, 4, 360, 640]
/model/enc1/enc1.0/conv/weight_quant/export_handler/Quant_output_0: [32, 4, 3, 3]
/model/enc1/enc1.0/relu/act_quant/export_handler/Quant_output_0: [1, 32, 180, 320]
/model/enc1/enc1.1/conv/weight_quant/export_handler/Quant_output_0: [32, 32, 3, 3]
/model/enc1/enc1.1/relu/act_quant/export_handler/Quant_output_0: [1, 32, 180, 320]
/model/enc2/enc2.0/conv/weight_quant/export_handler/Quant_output_0: [64, 32, 3, 3]
/model/enc2/enc2.0/relu/act_quant/export_handler/Quant_output_0: [1, 64, 90, 160]
/model/enc2/enc2.1/conv/weight_quant/export_handler/Quant_output_0: [64, 64, 3, 3]
/model/enc2/enc2.1/relu/act_quant/export_handler/Quant_output_0: [1, 64, 90, 160]
/model/enc3/enc3.0/conv/weight_quant/export_handler/Quant_output_0: [96, 64, 3, 3]
/model/enc3/enc3.0/relu/act_quant/export_handler/Quant_output_0: [1, 96, 45, 

## 4. Suche nach verdächtigen Dimensionen (z. B. 80 vs 45)

In [5]:
conflicts = []

def scan_shape(shape):
    return [d for d in shape if d in (80, 45)]

for vi in model_si.graph.value_info:
    shape = extract_shape(vi)
    hits = scan_shape(shape)
    if hits:
        conflicts.append((vi.name, shape))

print("🔎 Gefundene problematische Shapes (enthält 80 oder 45):")
pprint(conflicts)

🔎 Gefundene problematische Shapes (enthält 80 oder 45):
[('/model/enc3/enc3.0/relu/act_quant/export_handler/Quant_output_0',
  [1, 96, 45, 80]),
 ('/model/enc3/enc3.1/relu/act_quant/export_handler/Quant_output_0',
  [1, 96, 45, 80]),
 ('/model/bottleneck/bottleneck.0/relu/act_quant/export_handler/Quant_output_0',
  [1, 128, 45, 80]),
 ('/model/bottleneck/bottleneck.1/relu/act_quant/export_handler/Quant_output_0',
  [1, 96, 45, 80]),
 ('/model/decoder/decoder.0/relu/act_quant/export_handler/Quant_output_0',
  [1, 64, 45, 80]),
 ('/model/decoder/decoder.1/relu/act_quant/export_handler/Quant_output_0',
  [1, 32, 45, 80]),
 ('/model/enc3/enc3.0/conv/Conv_output_0', [1, 96, 45, 80]),
 ('/model/enc3/enc3.0/bn/BatchNormalization_output_0', [1, 96, 45, 80]),
 ('/model/enc3/enc3.0/relu/act_quant/activation_impl/Relu_output_0',
  [1, 96, 45, 80]),
 ('/model/enc3/enc3.1/conv/Conv_output_0', [1, 96, 45, 80]),
 ('/model/enc3/enc3.1/bn/BatchNormalization_output_0', [1, 96, 45, 80]),
 ('/model/enc3/e

## 5. Finde den Node `Transpose_3` und zeige seine Eingaben/Ausgaben

In [6]:
target = None
for n in model_si.graph.node:
    if n.name == "Transpose_3":
        target = n
        break

if target:
    print("Gefunden:")
    print("Name:", target.name)
    print("Inputs:", target.input)
    print("Outputs:", target.output)
else:
    print("❌ Node 'Transpose_3' wurde nicht gefunden.")

❌ Node 'Transpose_3' wurde nicht gefunden.
